# Category 4 - Embedding Model Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares embedding models for the Ally Vision v2 memory recall layer. The decision criteria are retrieval quality, multilingual fit, dimension flexibility, API price, and how naturally the model fits a DashScope-first stack.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Qwen embedding docs": "https://docs.qwencloud.com/developer-guides/embeddings/text-embedding",
    "Qwen pricing": "https://docs.qwencloud.com/developer-guides/getting-started/pricing",
    "OpenAI embeddings guide": "https://developers.openai.com/api/docs/guides/embeddings",
    "BAAI bge-m3 card": "https://huggingface.co/BAAI/bge-m3",
    "Project settings": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py"
  },
  "comparison_rows": [
    {
      "Metric": "MTEB overall score",
      "text-embedding-v4": "68.36 @ 1024 dims [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "63.39 @ 1024 dims [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "61.0 [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "62.3 [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "N/A (not publicly available) [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "62.28 @ 768 dims [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "MTEB retrieval score",
      "text-embedding-v4": "68.36 cached public value [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "63.39 cached public value [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "Strong multilingual retrieval references on HF card [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "N/A (not publicly available) [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "MTEB clustering score",
      "text-embedding-v4": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "N/A (not publicly available) [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "N/A (not publicly available) [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "MTEB STS score",
      "text-embedding-v4": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "N/A (not publicly available) [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "N/A (not publicly available) [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Dimensions supported",
      "text-embedding-v4": "2048,1536,1024,768,512,256,128,64 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "1024,768,512 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "1536 fixed [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "1024 [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "768 [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "1024 class public docs [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Max sequence length",
      "text-embedding-v4": "8192 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "8192 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "8192 [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "8192 [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "8192 class public positioning [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Language support count",
      "text-embedding-v4": "100+ languages [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "50+ languages [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "100+ languages [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "N/A (not publicly available) [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "Multilingual family [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "Multilingual family [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Matryoshka / flexible dims",
      "text-embedding-v4": "Yes [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "Limited dimension options [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "N/A (not publicly available) [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "Dimensional trade-off in model card [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "Task / mode variants [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Instruction tuned",
      "text-embedding-v4": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "Yes [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "N/A (not publicly available) [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "Yes [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "Yes [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Price per 1M tokens",
      "text-embedding-v4": "$0.07 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "$0.07 (cached historical value) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "$0.10 [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "N/A (not publicly available) [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "N/A (not publicly available) [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "N/A (not publicly available) [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "N/A (not publicly available) [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    },
    {
      "Metric": "Self-hostable availability",
      "text-embedding-v4": "No [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-v3": "N/A (not publicly available) [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
      "text-embedding-ada-002": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
      "text-embedding-3-small": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/guides/embeddings]",
      "bge-m3": "Yes [source: https://huggingface.co/BAAI/bge-m3]",
      "nomic-embed-text-v1.5": "Yes [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]",
      "cohere-embed-v3": "N/A (not publicly available) [source: https://docs.cohere.com/docs/embeddings]",
      "jina-embeddings-v3": "N/A (not publicly available) [source: https://jina.ai/embeddings/]",
      "e5-mistral-7b-instruct": "Yes [source: https://huggingface.co/intfloat/e5-mistral-7b-instruct]",
      "voyage-3": "N/A (not publicly available) [source: https://docs.voyageai.com/docs/embeddings]",
      "gte-Qwen2-7B": "Yes [source: https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct]"
    }
  ],
  "criteria": [
    "quality",
    "multilingual",
    "cost",
    "stack fit"
  ],
  "fit_matrix": {
    "text-embedding-v4": [
      1,
      1,
      1,
      1
    ],
    "text-embedding-v3": [
      0.7,
      0.7,
      1,
      0.8
    ],
    "text-embedding-ada-002": [
      0.6,
      0.4,
      0.5,
      0
    ],
    "text-embedding-3-small": [
      0.65,
      0.5,
      0,
      0
    ],
    "bge-m3": [
      0.7,
      1,
      1,
      0
    ],
    "nomic-embed-text-v1.5": [
      0.65,
      0.5,
      1,
      0
    ],
    "cohere-embed-v3": [
      0.6,
      0.7,
      0,
      0
    ],
    "jina-embeddings-v3": [
      0.6,
      0.6,
      0,
      0
    ],
    "e5-mistral-7b-instruct": [
      0.7,
      0.5,
      1,
      0
    ],
    "voyage-3": [
      0.6,
      0.6,
      0,
      0
    ],
    "gte-Qwen2-7B": [
      0.7,
      0.7,
      1,
      0
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
labels=list(data['fit_matrix'].keys())
values=[sum(v) for v in data['fit_matrix'].values()]
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 4 - Embedding Model Comparison Overall Fit Score")
ax.bar(labels,values,color=colors); ax.set_ylabel('Derived fit score',color='white'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(charts_dir / 'category4_embedding_model_comparison_chart1.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["text-embedding-v4", "text-embedding-v3", "bge-m3", "nomic-embed-text-v1.5", "e5-mistral-7b-instruct"]; x=np.arange(len(criteria)); width=0.8/len(selected)
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 4 - Embedding Model Comparison Top-5 Criteria View")
for idx,label in enumerate(selected):
 vals=data['fit_matrix'][label]; color=ALLY if label==selected[0] else (DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else COMP); ax.bar(x+(idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white'); plt.tight_layout(); plt.savefig(charts_dir / 'category4_embedding_model_comparison_chart2.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["text-embedding-v4", "text-embedding-v3", "bge-m3", "nomic-embed-text-v1.5", "e5-mistral-7b-instruct"]; mat=np.array([data['fit_matrix'][k] for k in selected])
fig,ax=plt.subplots(figsize=(10,5)); ax.set_facecolor(BG); ax.figure.set_facecolor(BG); im=ax.imshow(mat,cmap='Blues',aspect='auto'); ax.set_title('Ally Vision v2 — Category 4 - Embedding Model Comparison Trade-off Heatmap',color='white'); ax.set_xticks(np.arange(len(criteria))); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.set_yticks(np.arange(len(selected))); ax.set_yticklabels(selected,color='white')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]): ax.text(j,i,f"{mat[i,j]:.0f}",ha='center',va='center',color='white')
plt.colorbar(im); plt.tight_layout(); plt.savefig(charts_dir / 'category4_embedding_model_comparison_chart3.png',dpi=150,bbox_inches='tight'); plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Qwen embedding docs | https://docs.qwencloud.com/developer-guides/embeddings/text-embedding | MTEB, dimension, language support cache |
| 2 | Qwen pricing | https://docs.qwencloud.com/developer-guides/getting-started/pricing | embedding token pricing |
| 3 | OpenAI embeddings guide | https://developers.openai.com/api/docs/guides/embeddings | OpenAI embedding comparison values |
| 4 | BAAI bge-m3 card | https://huggingface.co/BAAI/bge-m3 | multilingual and retrieval metadata |
| 5 | Project settings | https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py | current text-embedding-v4 choice at 1024 dims |


## CONCLUSION

text-embedding-v4 is still the best fit for Ally Vision v2 because it combines the strongest cached public score in this comparison with broad language support, flexible dimensions, low price, and native DashScope integration. The project’s current 1024-dimension configuration is a practical balance between quality and storage cost.

→ Chosen for Ally Vision v2 ✅
